In [ ]:
import json
import random
import uuid

from pgvector.psycopg import register_vector

from model_endpoint_code import utils
from storage.postgres_client_wrapper import build_pg_connection, PostgresClientWrapper
from storage.vector_search import search_images_by_text, search_faces_by_embedding, find_best_images
from queries import INSERT_IMAGE, INSERT_FACE

CLIP_DIM = 512
FACE_DIM = 512

In [ ]:
creds = utils.PostgresCredentials(
    host="localhost",
    port=5432,
    database="postgres",
    user="postgres",
    password="password",
)

conn = build_pg_connection(creds)
register_vector(conn)
db = PostgresClientWrapper(conn)
print("connected")

## Insert Data

In [ ]:
def random_vec(dim: int) -> list[float]:
    return [random.random() for _ in range(dim)]


def insert_test_image(db, clip_embedding, uri="s3://bucket/test.jpg") -> str:
    image_id = str(uuid.uuid4())
    db.insert(
        INSERT_IMAGE,
        {"image_id": image_id, "image_uri": uri, "clip_embedding": clip_embedding},
    )
    return image_id


def insert_test_face(db, image_id, face_embedding, bbox=(0, 0, 10, 10)) -> str:
    face_id = str(uuid.uuid4())
    db.insert(
        INSERT_FACE,
        {
            "face_id": face_id,
            "image_id": image_id,
            "face_embedding": face_embedding,
            "bbox": json.dumps(list(bbox)),
        },
    )
    return face_id

In [ ]:
image_ids = []
for i in range(3):
    img_id = insert_test_image(db, random_vec(CLIP_DIM), uri=f"s3://bucket/img_{i}.jpg")
    image_ids.append(img_id)
    for _ in range(2):
        insert_test_face(db, img_id, random_vec(FACE_DIM))

print("inserted image_ids:", image_ids)

## Test Search Queries

In [ ]:
image_hits = search_images_by_text(db, random_vec(CLIP_DIM), top_k=5)
image_hits

In [ ]:
face_hits = search_faces_by_embedding(db, random_vec(FACE_DIM), top_k=5)
face_hits

## Test retrieval pipeline

In [ ]:
results = find_best_images(
    db,
    text_embedding=random_vec(CLIP_DIM),
    face_embedding=random_vec(FACE_DIM),
    top_k=5,
)
results 

## Clean Up

In [ ]:
with db._conn.cursor() as cur:
    cur.execute("DELETE FROM images WHERE image_id = ANY(%s)", (image_ids,))
print("cleaned up", image_ids)
